# TREAD-ONE NB04 — richer evidence and optimizer-recovery gate

**Engineering test only. It does not launch another 60-epoch study, modify r1
checkpoints, choose a new model or replace the workstation models.**

This tests real TREAD-ONE optimizer updates with richer direct CSV, actual
augmented-image predictions/targets, numerical/gradient/activation/resource
records, native maps and full optimizer/scheduler/AMP/RNG recovery state.

## Settings and run order

1. Kaggle **T4 ×2**, Internet **ON**. Both available T4 devices run independent
   compute-only recovery tests; one parent coordinates HF operations.
2. Attach `shanmuk4622/tire-dataset-prepared-phase2` directly from Kaggle.
3. Enable Kaggle Secret **HF_TOKEN**, write access to `Shanmuk4622/tyre-wear-study`.
4. Keep at least **12 GiB free** in working storage. The input dataset is not copied.
5. Run All. Existing packages are retained; no global dependency downgrade.

Each engineering case stops at **three** updates, crossing a declared two-batch
surrogate epoch boundary. The gate compares frozen r1 update math, uninterrupted
instrumented updates, fresh-process pause/resume, an abrupt native-process exit,
and reconstruction from directly stored snapshots. These repeated fixtures are
recovery evidence, not independent model experiments or training-data growth.

HF proof uploads a one-update state, downloads that pinned commit to a new
recovery base, continues to update three, and checks exact numerical state
against the uninterrupted run. Direct CSV and native `.pt`/`.npy` files share a
manifest and committed cursor; no ZIP/NPZ is published.

## Stop and resume

Use this notebook unchanged to resume. Compatible saved cases skip completed
updates. Local native recovery states rotate to the latest two; **all CSV and
declared gate maps remain retained**. HF commits preserve immutable history.
The parent attempts a consistent snapshot on catchable stop/error, major stage
completion and every 30 minutes, respecting server backoff. A forced kernel or
container death cannot flush unsent work. Only one parent may own a namespace;
clean completion releases its 45-minute writer lease.

The pass result validates this engineered logging/recovery path. It is not a
claim of long-run model quality, continuous GPU energy measurement, universal
network reliability, or a completed r2 study. A measured recipe and bounded map
schedule must be frozen before any later long training run.


In [1]:
import base64,gzip,json,os,signal,subprocess,sys,time
from pathlib import Path
from kaggle_secrets import UserSecretsClient
WORK=Path('/kaggle/working/treadone_evidence_gate')
WORK.mkdir(parents=True,exist_ok=True)
os.environ['HF_TOKEN']=UserSecretsClient().get_secret('HF_TOKEN')
sources=json.loads(gzip.decompress(base64.b64decode('...')))
for name,value in sources.items():
    assert '..' not in Path(name).parts and not Path(name).is_absolute()
    target=WORK/name;target.parent.mkdir(parents=True,exist_ok=True)
    target.write_bytes(base64.b64decode(value))
print('TREAD-ONE evidence gate ready. HF_TOKEN loaded without displaying it.')

def launch():
    child=subprocess.Popen([sys.executable,'-B','-u',str(WORK/'training/gate.py'),'--publish'],start_new_session=True)
    try:
        code=child.wait()
    except KeyboardInterrupt:
        print('Stop requested: waiting for completed-update state/CSV snapshot and HF flush.')
        os.killpg(child.pid,signal.SIGINT)
        for i in range(48):
            try:
                child.wait(timeout=5);break
            except subprocess.TimeoutExpired:
                if i%4==0:print('Still waiting for a safe snapshot or server backoff; keep the session open.')
        if child.poll() is None:print('The coordinator is still flushing; do not launch a duplicate writer.')
        raise
    if code:raise RuntimeError('Gate failed or paused. Read the traceback and retained coordinator.log; rerun unchanged to recover compatible cases.')


TREAD-ONE evidence gate ready. HF_TOKEN loaded without displaying it.


## Run the real-model gate
No old training bundle or pretrained model download is required. The prepared Kaggle input supplies all images/labels.

In [2]:
launch()

Running engineering gate on cuda:0 protocol acbfc1f71305b36d07d5ed24b1d03e91e37f57c1cc02653e6b5b71e4fd62c9ce
Running engineering gate on cuda:1 protocol da71929d8ae2443fdb334cd59cbfcb729e75ef856e68a76a321829b682caa388


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            
New Data Upload               : |          |  0.00B /  0.00B            

{
  "status": "passed",
  "devices": [
    "cuda:0",
    "cuda:1"
  ],
  "live_hf_tested": true,
  "engineering_only": true
}


## Inspect the gate and publication receipts
Pass/fail concerns engineering correctness. These few fixture losses are not study results.

In [3]:
from IPython.display import display,FileLink
result=json.loads((WORK/'training_gates/GATE_COMPLETION.json').read_text())
assert result['status']=='passed'
print(json.dumps(result,indent=2))
for device in result['devices']:
    folder=Path(device['path'])
    snapshot=json.loads((folder/'LATEST_ENVELOPE.json').read_text())['path']
    display(FileLink(str(Path(snapshot)/'cases/live_resume/EVIDENCE_GATE.html')))
    print('Verified HF receipt:',device['publication'])
print('Next: review the frozen logging schema and choose one controlled model/objective change; no new study was started by NB04.')


{
  "devices": [
    {
      "device": "cuda:0",
      "live": {
        "continued_to_updates": 3,
        "differences": {
          "losses": 0.0,
          "model": 0.0,
          "next_cursor": 0,
          "next_epoch": 0,
          "optimizer": 0.0,
          "rng": 0,
          "scaler": 0.0,
          "scheduler": 0,
          "updates": 0
        },
        "engineering_only": true,
        "published_source_revision": "87d3a9605006ab1c35d5779eb1ea456c9803eb89",
        "restored_completed_updates": 1,
        "row_cursors": {
          "activation_modules": 40,
          "class_confusion": 27,
          "epoch_summaries": 1,
          "events": 3,
          "evidence_metrics": 72,
          "geometry_points": 144,
          "gradient_modules": 48,
          "latent_metrics": 288,
          "region_stats": 48,
          "resources": 5,
          "train_samples": 24,
          "train_steps": 3,
          "validation_predictions": 2
        },
        "status": "passed"
      }

/kaggle/working/treadone_evidence_gate/training_gates/acbfc1f71305b36d07d5ed24b1d03e91e37f57c1cc02653e6b5b71e4fd62c9ce/envelopes/passed-bfa9ed078b/cases/live_resume/EVIDENCE_GATE.html

Verified HF receipt: {'destination': 'phase3/tread-one/engineering/evidence-gate-r1/acbfc1f71305b36d07d5ed24b1d03e91e37f57c1cc02653e6b5b71e4fd62c9ce', 'files': 285, 'owner_released': True, 'revision': '6922039ee7a5bb0ea75c9f0827c650aea0c99807', 'status': 'published_and_verified', 'zip_uploads': 0}


/kaggle/working/treadone_evidence_gate/training_gates/da71929d8ae2443fdb334cd59cbfcb729e75ef856e68a76a321829b682caa388/envelopes/passed-85008460eb/cases/live_resume/EVIDENCE_GATE.html

Verified HF receipt: {'destination': 'phase3/tread-one/engineering/evidence-gate-r1/da71929d8ae2443fdb334cd59cbfcb729e75ef856e68a76a321829b682caa388', 'files': 287, 'owner_released': True, 'revision': '2878662661a364afc06be0627128adfd0d197b13', 'status': 'published_and_verified', 'zip_uploads': 0}
Next: review the frozen logging schema and choose one controlled model/objective change; no new study was started by NB04.
